# Molécula de hidrogênio $H_2$

Este notebook apresenta a simulação quântica da molécula de hidrogênio ($H_2$)
utilizando o algoritmo Variational Quantum Eigensolver (VQE).

O objetivo é estimar a energia do estado fundamental da molécula a partir
da construção do Hamiltoniano eletrônico e sua posterior otimização variacional.

In [ ]:
from src.hamiltonian import build_qubit_hamiltonian, get_atom_energy
from src.ansatz import build_ansatz
from src.optimizer import get_optimizer
from src.vqe_runner import run_vqe, run_experiment
from src.results import save_history, save_summary
from src.plots import compute_dissociation_curve

import numpy as np

In [ ]:
from qiskit_algorithms.utils import algorithm_globals

algorithm_globals.random_seed = 137

## Definição do sistema molecular

Nesta etapa definimos a geometria da molécula de hidrogênio.
A distância interatômica escolhida corresponde aproximadamente
ao comprimento de ligação de equilíbrio.

In [ ]:
atom = "H 0 0 0; H 0 0 0.735"

## Aplicando o VQE

A partir da geometria molecular, construímos o Hamiltoniano quântico
em representação de qubits. Esse operador será utilizado como entrada
para o algoritmo VQE, responsável por aproximar a energia fundamental.

In [ ]:
data = build_qubit_hamiltonian(atom)
ansatz = build_ansatz(name="efficient_su2", num_qubits=data["num_qubits"])
optimizer = get_optimizer(max_iter=200)

result = run_vqe(data["qubit_op"], ansatz, optimizer)

##  Persistência dos dados

Os resultados obtidos são armazenados para posterior análise e
reprodutibilidade do experimento. Isso permite comparar execuções
com diferentes configurações variacionais ou moléculas.

In [ ]:
save_summary(
    "h2_summary.csv",
    "H2",
    result["energy"],
    data["fci_energy"],
    result["iterations"]
)

save_history("h2_history.csv", result["history"])


### Validando a saída

Por fim, verificamos a energia estimada pelo VQE e a comparamos
com o valor esperado para garantir a consistência da simulação.

In [ ]:
print("VQE:", result["energy"])
print("FCI:", data["fci_energy"])
print("Erro:", abs(result["energy"] - data["fci_energy"]))

## Curvas de energia

### Calcular energia do átomo H isolado

In [ ]:
E_H = get_atom_energy(atom_symbol="H", spin=1)
E_ref = 2 * E_H

### Plot da curva de dissociação

$$
E_{binding} = E(H_2) - 2 * E(H)
$$

In [ ]:
import warnings
from scipy.sparse import SparseEfficiencyWarning

warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)

In [ ]:
distances = np.linspace(0.5, 2.5, 10)
distances = np.sort(distances)

vqe_energies = []
fci_energies = []

initial_point = None

for d in distances:
    config = {
        "molecule": "H2",
        "geometry": f"H 0 0 0; H 0 0 {d}",
        "basis": "sto-3g",
        "ansatz": "uccsd",
        "max_iter": 500,
    }

    result = run_experiment(config, initial_point)

    initial_point = result["optimal_params"]
    
    vqe_energies.append(result["energy"] - E_ref)
    fci_energies.append(result["fci"] - E_ref)

# Conversão Hartree para kcal/mol
hartree_to_kcal = 627.509

vqe_energies = np.array(vqe_energies) * hartree_to_kcal
fci_energies = np.array(fci_energies) * hartree_to_kcal

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(distances, vqe_energies, "o--", label="VQE", linewidth=2, markersize=8)
plt.plot(distances, fci_energies, "-", label="FCI", linewidth=2, markersize=6)

plt.xlabel("Distância interatômica (Å)", fontsize=12)
plt.ylabel("Energia de dissociação (kcal/mol)", fontsize=12)
plt.title("Curva de dissociação do H₂ - STO-3G", fontsize=14)

plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

## Comparativo de bases

In [ ]:
bases = ["sto-3g", "6-31g", "cc-pvdz"]
distances = np.linspace(0.5, 2.5, 10)

plt.figure(figsize=(10, 6))

for basis in bases:
    config = {
        "molecule": "H2",
        "basis": basis,
        "ansatz": "uccsd",
        "max_iter": 200,
    }

    E_H = get_atom_energy("H", basis=basis, spin=1)
    E_ref = 2 * E_H

    vqe, fci = compute_dissociation_curve(config, distances, E_ref)

    plt.plot(distances, vqe, "o--", label=f"VQE ({basis})")
    plt.plot(distances, fci, "-", label=f"FCI ({basis})")

plt.xlabel("Distância (Å)")
plt.ylabel("Energia (kcal/mol)")
plt.title("Comparação entre bases - H₂")
plt.legend()
plt.grid()
plt.show()

## Comparativos de ansatz

In [ ]:
ansatz_list = ["efficient_su2", "uccsd"]
distances = np.linspace(0.5, 2.5, 10)

plt.figure(figsize=(10, 6))

for ansatz in ansatz_list:
    config = {
        "molecule": "H2",
        "basis": "sto-3g",
        "ansatz": ansatz,
        "max_iter": 200,
    }

    E_H = get_atom_energy("H", basis="sto-3g", spin=1)
    E_ref = 2 * E_H

    vqe, fci = compute_dissociation_curve(config, distances, E_ref)

    plt.plot(distances, vqe, "o--", label=f"VQE ({ansatz})")

plt.plot(distances, fci, "-", label="FCI")

plt.xlabel("Distância (Å)")
plt.ylabel("Energia (kcal/mol)")
plt.title("Comparação de ansatz - H₂")
plt.legend()
plt.grid()
plt.show()

## Teste PySCF puro

In [ ]:
from typing import Dict, Any
from pyscf import gto, scf, fci


def build_hamiltonian_pyscf(
    atom_string: str,
    basis: str = "sto-3g"
) -> Dict[str, Any]:

    # -------------------------
    # 1. Definir molécula
    # -------------------------
    mol = gto.Mole()
    mol.atom = atom_string
    mol.basis = basis
    mol.unit = "Angstrom"
    mol.spin = 0  # H2 = singlete
    mol.charge = 0
    mol.build()

    # -------------------------
    # 2. Hartree-Fock
    # -------------------------
    mf = scf.RHF(mol)
    mf.kernel()

    # -------------------------
    # 3. FCI (Full Configuration Interaction)
    # -------------------------
    ci_solver = fci.FCI(mol, mf.mo_coeff)
    energy, _ = ci_solver.kernel()

    return {
        "fci_energy": energy,
        "num_orbitals": mf.mo_coeff.shape[1],
        "num_electrons": mol.nelectron,
    }